In [ ]:
import requests
from bs4 import BeautifulSoup
import json, time
from tqdm import tqdm

BASE = "https://eparlib.sansad.in"
HEADERS = {"User-Agent": "Mozilla/5.0"}

with open("handles.json") as f:
    handles = json.load(f)

pdf_links = []

for item in tqdm(handles):
    try:
        r = requests.get(item["handle_url"], headers=HEADERS, timeout=10)

        soup = BeautifulSoup(r.text, "html.parser")

        for a in soup.find_all("a", href=True):
            href = a["href"]

            if "/bitstream/" in href and href.lower().endswith(".pdf"):
                pdf_links.append({
                    "date": item["date"],
                    "url": BASE + href if href.startswith("/") else href
                })
                break

        time.sleep(0.2)  # small delay only

    except:
        continue

with open("pdf_links.json", "w") as f:
    json.dump(pdf_links, f)

print("DONE:", len(pdf_links))

100%|██████████| 1280/1280 [36:30<00:00,  1.71s/it]

DONE: 1279


In [ ]:
!pip install pdfplumber requests beautifulsoup4 tqdm -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.1/68.1 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 56.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 70.9 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import requests, pdfplumber, json, time, os, re, io, glob
from tqdm import tqdm

OUTPUT_DIR = "/content/drive/MyDrive/sansad_data/"
os.makedirs(OUTPUT_DIR, exist_ok=True)

CHUNK_SIZE = 800
CHUNK_OVERLAP = 150
FILE_LIMIT = 15000

HEADERS = {"User-Agent": "Mozilla/5.0"}

# load pdf links
with open("/content/pdf_links.json") as f:
    pdf_links = json.load(f)

# detect processed files
processed_files = set()
chunk_files = glob.glob(f"{OUTPUT_DIR}/chunks_*.json")

for path in chunk_files:
    with open(path) as f:
        data = json.load(f)
    for item in data:
        processed_files.add(item["metadata"]["filename"])

print(f"Already processed: {len(processed_files)}")

# next chunk index
existing_indices = [
    int(os.path.basename(f).split("_")[1].split(".")[0])
    for f in chunk_files
]

file_index = max(existing_indices)+1 if existing_indices else 0
chunk_id = file_index * FILE_LIMIT

all_chunks = []

# helpers
def clean_text(text):
    text = re.sub(r'\n{3,}', '\n\n', text)
    text = re.sub(r'[ \t]{2,}', ' ', text)
    return text.strip()

def sliding_chunks(text):
    res, start = [], 0
    step = CHUNK_SIZE - CHUNK_OVERLAP
    while start < len(text):
        res.append(text[start:start+CHUNK_SIZE])
        start += step
    return res

def extract_meta(text):
    month_match = re.search(
        r'(January|February|March|April|May|June|July|August|September|October|November|December)',
        text, re.IGNORECASE)
    month = month_match.group().capitalize() if month_match else "unknown"

    m = month.lower()
    if m in ("january","february","march","april","may"):
        session = "Budget Session"
    elif m in ("june","july","august"):
        session = "Monsoon Session"
    elif m in ("november","december"):
        session = "Winter Session"
    else:
        session = "Special Session"

    return month, session


# main loop
for item in tqdm(pdf_links):

    filename = item["url"].split("/")[-1]

    if filename in processed_files:
        continue

    pdf_bytes = None

    for attempt in range(3):
        try:
            r = requests.get(item["url"], headers=HEADERS, timeout=40)
            if r.status_code == 200:
                pdf_bytes = io.BytesIO(r.content)
                break
        except:
            pass
        time.sleep(4)

    if pdf_bytes is None:
        continue

    try:
        with pdfplumber.open(pdf_bytes) as pdf:

            first_text = pdf.pages[0].extract_text() or ""
            month, session = extract_meta(first_text)

            for page_num, page in enumerate(pdf.pages, 1):

                try:
                    raw = page.extract_text()
                except:
                    continue

                if not raw or len(raw.strip()) < 50:
                    continue

                cleaned = clean_text(raw)

                for ci, chunk in enumerate(sliding_chunks(cleaned)):

                    if len(chunk.strip()) < 80:
                        continue

                    all_chunks.append({
                        "id": chunk_id,
                        "text": chunk,
                        "metadata": {
                            "filename": filename,
                            "month": month,
                            "session": session,
                            "page": page_num
                        }
                    })

                    chunk_id += 1

                    if len(all_chunks) >= FILE_LIMIT:
                        path = f"{OUTPUT_DIR}/chunks_{file_index}.json"
                        with open(path, "w") as f:
                            json.dump(all_chunks, f)

                        print("Saved:", path)

                        all_chunks = []
                        file_index += 1

    except:
        continue

    time.sleep(1.5)


# final save
if all_chunks:
    path = f"{OUTPUT_DIR}/chunks_{file_index}.json"
    with open(path, "w") as f:
        json.dump(all_chunks, f)

print("DONE")

Already processed: 793


 63%|██████▎   | 812/1279 [25:43<5:49:40, 44.93s/it]

Saved: /content/drive/MyDrive/sansad_data//chunks_63.json


 64%|██████▍   | 824/1279 [44:44<12:32:59, 99.30s/it] 

Saved: /content/drive/MyDrive/sansad_data//chunks_64.json


...

Saved: /content/drive/MyDrive/sansad_data//chunks_98.json


100%|█████████▉| 1276/1279 [9:03:22<02:06, 42.11s/it]

Saved: /content/drive/MyDrive/sansad_data//chunks_99.json


100%|██████████| 1279/1279 [9:05:51<00:00, 25.61s/it]

DONE


In [ ]:
!pip install faiss-cpu sentence-transformers -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 29.1 MB/s eta 0:00:00


In [ ]:
import json
import numpy as np
import faiss
import pickle
import glob
from tqdm import tqdm
from sentence_transformers import SentenceTransformer

# ── CONFIG ─────────────────────────────────────────────
DATA_DIR   = "/content/drive/MyDrive/sansad_data/"
INDEX_FILE = DATA_DIR + "faiss.index"
META_FILE  = DATA_DIR + "metadata.pkl"
EMBED_FILE = DATA_DIR + "embeddings.npy"

EMBED_MODEL = "all-MiniLM-L6-v2"
BATCH_SIZE  = 64


# ── LOAD ALL CHUNKS ────────────────────────────────────
chunk_files = sorted(glob.glob(DATA_DIR + "chunks_*.json"))

all_chunks = []

print(f"Loading {len(chunk_files)} chunk files...")

for file in chunk_files:
    with open(file) as f:
        data = json.load(f)
        all_chunks.extend(data)

print(f"✅ Total chunks loaded: {len(all_chunks)}")


# ── LOAD MODEL ─────────────────────────────────────────
print("Loading embedding model...")
model = SentenceTransformer(EMBED_MODEL)


# ── EMBED ──────────────────────────────────────────────
texts = [c["text"] for c in all_chunks]

embeddings = []

print("Embedding...")

for i in tqdm(range(0, len(texts), BATCH_SIZE)):
    batch = texts[i:i+BATCH_SIZE]
    vecs = model.encode(batch, normalize_embeddings=True)
    embeddings.append(vecs)

embeddings = np.vstack(embeddings).astype("float32")

print("✅ Embeddings shape:", embeddings.shape)


# ── SAVE EMBEDDINGS (IMPORTANT) ────────────────────────
np.save(EMBED_FILE, embeddings)
print(f"💾 Saved embeddings → {EMBED_FILE}")


# ── BUILD FAISS ───────────────────────────────────────
dim = embeddings.shape[1]

index = faiss.IndexFlatIP(dim)
index.add(embeddings)

print(f"✅ FAISS index built: {index.ntotal} vectors")


# ── SAVE INDEX ────────────────────────────────────────
faiss.write_index(index, INDEX_FILE)
print(f"💾 Saved FAISS → {INDEX_FILE}")


# ── SAVE METADATA ─────────────────────────────────────
metadata = [c["metadata"] | {"text": c["text"]} for c in all_chunks]

with open(META_FILE, "wb") as f:
    pickle.dump(metadata, f)

print(f"💾 Saved metadata → {META_FILE}")

print("\n🎉 DONE — Your vector DB is ready!")

Loading 101 chunk files...
✅ Total chunks loaded: 1501518
Loading embedding model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding...


  1%|▏         | 319/23462 [37:21<45:09:42,  7.03s/it]


KeyboardInterrupt: 